In [ ]:
import os
import json
import itertools
import pandas as pd
import torch
from PIL import Image
from diffusers.utils import make_image_grid
from IPython.display import display, clear_output
from transformers import CLIPTextModel, T5EncoderModel, BitsAndBytesConfig
from diffusers import AutoencoderKL
from diffusers.models import FluxTransformer2DModel


%cd ..
from slideredit.pipelines import SliderEditFluxKontextPipeline
from slideredit.pipelines import LoRAAdapterType

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

c:\Users\oikoh\1Coding\Thesis
CUDA Available: True
Device Name: NVIDIA GeForce RTX 5060 Laptop GPU


## HuggingFace

In [ ]:
from huggingface_hub import login

token_path = "/cs/student/msc/ml/2025/eoikonom/.hf_token"
if os.path.exists(token_path):
    with open(token_path, "r") as f:
        login(token=f.read().strip())

# Load FLUX

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

MODEL_ID = "black-forest-labs/FLUX.1-Kontext-dev"

# Load heavy text encoders straight to GPU in bfloat16
clip_text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=torch.bfloat16).to("cuda")
t5_text_encoder = T5EncoderModel.from_pretrained(MODEL_ID, subfolder="text_encoder_2", torch_dtype=torch.bfloat16).to("cuda")
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae").to("cuda", dtype=torch.bfloat16)

transformer = FluxTransformer2DModel.from_pretrained(
    MODEL_ID,
    subfolder="transformer",
    quantization_config=quantization_config,
    device_map={"": "cuda"}
)

print("🔌 Assembling the SliderEdit Pipeline...")
pipe = SliderEditFluxKontextPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    text_encoder=clip_text_encoder,
    text_encoder_2=t5_text_encoder,
    transformer=transformer,
    torch_dtype=torch.bfloat16
)

print(f"Loaded {MODEL_ID} successfully!")

### Load CSV

In [ ]:
CSV_PATH = "./experiments/experiments24-7.csv"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Cannot find {CSV_PATH}. Please ensure it is pushed to Git.")

df_experiments = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df_experiments)} experiments from {CSV_PATH}\n")
display(df_experiments[["id", "domain", "subprompt_1", "subprompt_2", "test_focus"]].head(10))

### Run Experiments Function Using CSV

In [ ]:
def run_benchmark_suite(
    df, 
    pipe, 
    mode="gstlora",  # Options: 'gstlora' (1D single slider) or 'stlora' (2D grid)
    checkpoint_path=None,  # Optional path to a LoRA checkpoint to load
    alpha_steps=[1.0, 0.75, 0.50, 0.25, 0.0],
    output_root="./outputs",
    source_img_dir="./datasets/test_images"
):
    """
    Iterates through the benchmark DataFrame, generates or loads base source images,
    runs 1D or 2D SliderEdit sweeps, saves metadata, and outputs image grids.
    """
    os.makedirs(source_img_dir, exist_ok=True)
    mode_output_dir = os.path.join(output_root, mode)
    os.makedirs(mode_output_dir, exist_ok=True)

    # Set appropriate adapter mode in pipeline
    if mode == "gstlora":
        if checkpoint_path:
            pipe.load_gstlora(checkpoint_path)
        pipe.loaded_adapter = LoRAAdapterType.GSTLORA
    elif mode == "stlora":
        if checkpoint_path:
            pipe.load_stlora(checkpoint_path, lora_rank=16, lora_dropout=0.0)
        pipe.loaded_adapter = LoRAAdapterType.STLORA
    else:
        raise ValueError("mode must be 'gstlora' or 'stlora'")

    print(f"==================================================")
    print(f"  STARTING BENCHMARK RUN: {mode.upper()}")
    print(f"  Alpha Range: {alpha_steps}")
    print(f"==================================================\n")

    for idx, row in df.iterrows():
        sweep_id = row['id']
        subprompt1 = str(row['subprompt_1'])
        subprompt2 = str(row['subprompt_2'])
        seed = int(row['seed'])
        base_prompt = str(row['base_prompt'])
        test_focus = str(row['test_focus'])

        # Create experiment subfolder: ./outputs/<mode>/<sweep_id>/
        exp_dir = os.path.join(mode_output_dir, sweep_id)
        os.makedirs(exp_dir, exist_ok=True)

        # ----------------------------------------------------------------------
        # Step 1: Ensure Base Source Image Exists (Generate via FLUX if missing)
        # ----------------------------------------------------------------------
        source_img_path = os.path.join(source_img_dir, f"{sweep_id}.png")
        if os.path.exists(source_img_path):
            img = Image.open(source_img_path).convert("RGB")
        else:
            print(f"[{sweep_id}] Source image missing. Generating base image from prompt...")

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                # temporarily disable adapters for clean base generation (!!!)
                with pipe.disable_adapters():
                    img = pipe(
                        prompt=base_prompt,
                        generator=torch.Generator(device="cuda").manual_seed(seed)
                    ).images[0]
            img.save(source_img_path)
            print(f"Saved base image to {source_img_path}")

        # ----------------------------------------------------------------------
        # Step 2: Save metadata.json for Offline Metric Evaluation
        # ----------------------------------------------------------------------
        meta_data = {
            "id": sweep_id,
            "domain": row['domain'],
            "mode": mode,
            "base_prompt": base_prompt,
            "subprompt_1": subprompt1,
            "subprompt_2": subprompt2,
            "seed": seed,
            "alpha_steps": alpha_steps,
            "test_focus": test_focus
        }
        with open(os.path.join(exp_dir, "meta.json"), "w") as f:
            json.dump(meta_data, f, indent=4)

        # ----------------------------------------------------------------------
        # Step 3: Run Inference Sweeps
        # ----------------------------------------------------------------------
        outputs = []
        
        if mode == "gstlora":
            # --- 1D Single Slider Sweep (using subprompt_1) ---
            edit_prompt = subprompt1
            print(f"\n[{idx+1}/{len(df)}] Running 1D gSTLoRA Sweep '{sweep_id}': '{edit_prompt}'")

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                for a in alpha_steps:
                    out_img = pipe(
                        image=img,
                        prompt=edit_prompt,
                        generator=torch.Generator(device="cuda").manual_seed(seed),
                        slider_alpha=a
                    ).images[0]

                    clean_prompt = "".join(c for c in edit_prompt if c.isalnum() or c in (" ", "_")).replace(" ", "_")
                    fname = f"{sweep_id}_{clean_prompt}_alpha_{a:.2f}.png"
                    out_img.save(os.path.join(exp_dir, fname))
                    outputs.append(out_img)

            # Display 1D Horizontal Row
            clear_output(wait=True)
            print(f"Completed [{sweep_id}] - Focus: {test_focus}")
            grid = make_image_grid([x.resize((128, 128)) for x in outputs], rows=1, cols=len(alpha_steps))
            display(grid)

        elif mode == "stlora":
            # --- 2D Multi Slider Sweep (subprompt_1 AND subprompt_2) ---
            subprompts_list = [subprompt1, subprompt2]
            edit_prompt = f"{subprompt1} and {subprompt2}"
            alpha_pairs = list(itertools.product(alpha_steps, repeat=2))
            
            print(f"\n[{idx+1}/{len(df)}] Running 2D STLoRA Grid '{sweep_id}': '{edit_prompt}'")

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                for a1, a2 in alpha_pairs:
                    out_img = pipe(
                        image=img,
                        prompt=edit_prompt,
                        generator=torch.Generator(device="cuda").manual_seed(seed),
                        subprompts_list=subprompts_list,
                        slider_alpha_list=[a1, a2]
                    ).images[0]

                    clean1 = "".join(c for c in subprompt1 if c.isalnum() or c in (" ", "_")).replace(" ", "_")
                    clean2 = "".join(c for c in subprompt2 if c.isalnum() or c in (" ", "_")).replace(" ", "_")
                    fname = f"{sweep_id}_{clean1}_{a1:.2f}_{clean2}_{a2:.2f}.png"
                    out_img.save(os.path.join(exp_dir, fname))
                    outputs.append(out_img)

            # Display 2D Grid
            clear_output(wait=True)
            print(f"Completed [{sweep_id}] - Focus: {test_focus}")
            grid = make_image_grid([x.resize((128, 128)) for x in outputs], rows=len(alpha_steps), cols=len(alpha_steps))
            display(grid)

        # memory cleanup after each experiment
        del outputs
        del grid
        torch.cuda.empty_cache()

    print("\nBenchmark Run Complete!")

In [ ]:
# reverse direction for better visualization of edits (1.0 = full suppression, 0.0 = full edit)
SWEEP_ALPHAS = [1.0, 0.75, 0.50, 0.25, 0.0]

run_benchmark_suite(
    df=df_experiments, 
    pipe=pipe, 
    mode="gstlora", 
    checkpoint_path="./checkpoints/gstlora_iter500.safetensors",
    alpha_steps=SWEEP_ALPHAS
)

In [ ]:
GRID_ALPHAS = [0.0, 0.50, 1.00]

run_benchmark_suite(
    df=df_experiments, 
    pipe=pipe, 
    mode="stlora", 
    checkpoint_path="./checkpoints/stlora_iter1200.pt",
    alpha_steps=GRID_ALPHAS
)